In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

from matplotlib.colors import ListedColormap
from sklearn.datasets import load_diabetes, load_iris, load_wine
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_curve,
    roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
    plot_tree
)
from typing import List, Optional, Tuple, Union
from xgboost import XGBClassifier, XGBRegressor

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader_simple import load_and_preprocess

df = load_and_preprocess(PROJECT_ROOT / "data" / "raw" / "dataset_practica_final.csv")
X = df.drop(columns="is_canceled")
y = df["is_canceled"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train.copy()
X_test = X_test.copy()

LOG_COLUMNS = [
    "lead_time",
    "days_in_waiting_list",
    "previous_cancellations",
    "previous_bookings_not_canceled",
]

log_columns = [c for c in LOG_COLUMNS if c in X_train.columns]
X_train[log_columns] = np.log1p(X_train[log_columns])
X_test[log_columns]  = np.log1p(X_test[log_columns])

CONTINUOUS_COLUMNS = [
    "lead_time",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "adr_per_person",
    "adr_per_night",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "total_guests",
    "total_nights",
]

scaled_columns = [column for column in CONTINUOUS_COLUMNS if column in X_train.columns]
scaler = StandardScaler()
X_train[scaled_columns] = scaler.fit_transform(X_train[scaled_columns])
X_test[scaled_columns] = scaler.transform(X_test[scaled_columns])

In [6]:
dict_parametros = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear']
}


rl_model = LogisticRegression(max_iter=500, random_state=42)
rl_model_grid_search = GridSearchCV(estimator=rl_model, param_grid=dict_parametros, cv=5, scoring='f1')
rl_model_grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=500, random_state=42),
             param_grid={'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear']},
             scoring='f1')

In [5]:
print(f"Mejores hiperparámetros encontrados: {rl_model_grid_search.best_params_}")
print(f"Mejor score obtenido: {rl_model_grid_search.best_score_:.2%}")

Mejores hiperparámetros encontrados: {'C': 10, 'solver': 'liblinear'}
Mejor score obtenido: 81.12%
